# Prática 3 — Roteamento manual

Heurística + LLM judge, sem LiteLLM.

## Fluxo de roteamento

```mermaid
flowchart LR
    U[Mensagem na conversa] --> H[Heurística de baixo custo]
    H --> D{Classificação clara?}

    D -- Simples --> LOCAL[Modelo local / menor]
    D -- Complexa --> STRONG[Modelo forte]
    D -- Ambígua --> J[LLM judge]
    J --> R{Rota escolhida}
    R -- Local --> LOCAL
    R -- Forte --> STRONG

    LOCAL --> RESP[Resposta]
    STRONG --> RESP
    J --> JM[Tokens e custo do judge]
    RESP --> COST[Consolidação de custos]
    JM --> COST
    COST --> OUT[Custo total e<br/>economia percentual]

    classDef low fill:#eaf7ee,stroke:#16883e,stroke-width:2px;
    classDef high fill:#fdecec,stroke:#b42318,stroke-width:2px;
    class LOCAL,H low;
    class STRONG high;
```

A heurística resolve os casos óbvios; o judge só é chamado quando a rota é ambígua, e seu custo também entra no consolidado.

In [1]:
# 0. Instalação reproduzível — propositalmente sem LiteLLM
import subprocess, sys
if subprocess.run([sys.executable, "-m", "pip", "--version"], capture_output=True).returncode:
    subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--only-binary=:all:",
    "python-dotenv==1.2.4", "openai==3.57.0", "pandas==3.1.1"
])
print("Bibliotecas instaladas. Se foi a primeira execução, reinicie o kernel.")

Bibliotecas instaladas. Se foi a primeira execução, reinicie o kernel.


In [2]:
# 1. Carrega e valida somente variáveis do .env
from pathlib import Path
import os
from dotenv import load_dotenv

ROOT = Path.cwd() if (Path.cwd() / ".env").is_file() else Path.cwd().parent
load_dotenv(ROOT / ".env", override=True)

required = ["OCI_GENAI_API_KEY", "OCI_GENAI_BASE_URL", "OCI_GENAI_MODEL"]
missing = [x for x in required if not os.getenv(x)]
if missing: raise ValueError("Preencha no .env: " + ", ".join(missing))

LOCAL_MODEL = os.getenv("LOCAL_LLM_MODEL", "openai.gpt-oss-20b")
STRONG_MODEL = os.getenv("SOL_MODEL", "openai.gpt-6.2-sol")
JUDGE_MODEL = os.getenv("ROUTER_JUDGE_MODEL", os.environ["OCI_GENAI_MODEL"])
print("Modelos:", {"local": LOCAL_MODEL, "forte": STRONG_MODEL, "judge": JUDGE_MODEL})

Modelos: {'local': 'openai.gpt-oss-20b', 'forte': 'openai.gpt-6.2-sol', 'judge': 'openai.gpt-5.6-sol'}


## 2. Preços ilustrativos e editáveis no .env

In [3]:
# USD por 1 milhão de tokens. Troque pelos valores do seu contrato.
PRICES = {
    LOCAL_MODEL: (float(os.getenv("LOCAL_INPUT_USD_MTOK", "0.10")),
                  float(os.getenv("LOCAL_OUTPUT_USD_MTOK", "0.40"))),
    STRONG_MODEL: (float(os.getenv("SOL_INPUT_USD_MTOK", "1.75")),
                   float(os.getenv("SOL_OUTPUT_USD_MTOK", "14.00"))),
    JUDGE_MODEL: (float(os.getenv("JUDGE_INPUT_USD_MTOK", "0.10")),
                  float(os.getenv("JUDGE_OUTPUT_USD_MTOK", "0.40"))),
}

def token_cost(model, input_tokens, output_tokens):
    input_price, output_price = PRICES[model]
    return input_tokens / 1_000_000 * input_price + output_tokens / 1_000_000 * output_price

PRICES

{'openai.gpt-oss-20b': (0.1, 0.4),
 'openai.gpt-6.2-sol': (1.75, 14.0),
 'openai.gpt-5.6-sol': (0.1, 0.4)}

## 3. Uma conversa com tarefas simples e complexas

In [4]:
MESSAGES = [
    "Cumprimente a cliente Marina.",
    "Resuma em uma frase: o painel está lento desde ontem.",
    "Qual é a capital de Santa Catarina?",
    "Reescreva com tom cordial: precisamos de mais detalhes.",
    "Traduza para inglês: chamado resolvido com sucesso.",
    "Classifique como positivo ou negativo: o atendimento foi excelente.",
    "Resuma em cinco palavras: a integração voltou a funcionar.",
    "Cumprimente o cliente João e deseje um bom dia.",
    "Explique em duas frases o que é cache.",
    "Sugira um título curto para um relatório de atendimento.",
    "Compare RAG e fine-tuning para suporte regulado e recomende uma arquitetura.",
    "Crie uma query SQL com CTE e janela para detectar anomalias por cliente.",
    "Analise o risco de migrar este agente sem testes de regressão.",
    "Monte um plano de diagnóstico para latência intermitente em produção.",
    "Defina uma estratégia de migração gradual para uma arquitetura multiagente.",
    "Compare três arquiteturas de memória e recomende uma para dados sensíveis.",
    "Escreva código Python para retentativas exponenciais com circuit breaker.",
    "Analise os riscos financeiros e operacionais desta automação.",
    "Otimize uma query SQL que consulta milhões de eventos por cliente.",
    "Crie um plano de testes para validar o agente antes da produção.",
]
print(f"{len(MESSAGES)} mensagens: tarefas simples, ambíguas e complexas.")

20 mensagens: tarefas simples, ambíguas e complexas.


## 4. Cliente LLM e medição real de tokens e custo

In [5]:
from dataclasses import dataclass
from openai import OpenAI

client = OpenAI(api_key=os.environ["OCI_GENAI_API_KEY"],
                base_url=os.environ["OCI_GENAI_BASE_URL"], timeout=90)

@dataclass
class Call:
    text: str
    input_tokens: int
    output_tokens: int
    cost_usd: float

def call_model(model, prompt, system="Responda com objetividade."):
    response = client.responses.create(model=model, instructions=system, input=prompt)
    return Call(response.output_text, response.usage.input_tokens, response.usage.output_tokens,
                token_cost(model, response.usage.input_tokens, response.usage.output_tokens))

def conversation_prompt(previous_messages, current):
    history = "\n".join(f"Usuário: {x}" for x in previous_messages) or "(início)"
    return f"Histórico da conversa:\n{history}\n\nTarefa atual:\n{current}"

## 5. Cenário base — tudo vai para o modelo forte

In [6]:
baseline_rows, history = [], []
for message in MESSAGES:
    result = call_model(STRONG_MODEL, conversation_prompt(history, message))
    baseline_rows.append({
        "cenário": "Sem roteamento", "mensagem": message, "rota": STRONG_MODEL,
        "motivo": "todas as tarefas usam o modelo forte",
        "input_tokens": result.input_tokens, "output_tokens": result.output_tokens,
        "custo_modelo_usd": result.cost_usd,
        "custo_judge_usd": 0.0, "tokens_judge": 0,
    })
    history.append(message)
print(f"Cenário base concluído: {len(baseline_rows)} chamadas no modelo forte.")

Cenário base concluído: 20 chamadas no modelo forte.


## 6. Roteador simples: heurística primeiro, LLM judge só em caso ambíguo

In [7]:
import json, re

COMPLEX = {"arquitetura", "sql", "risco", "riscos", "migrar", "migração",
           "diagnóstico", "produção", "compare", "recomende", "anomalias",
           "plano", "código", "otimize", "financeiros", "multiagente"}
SIMPLE = {"cumprimente", "resuma", "capital", "reescreva", "traduza", "classifique"}

def heuristic(message):
    words = set(re.findall(r"\w+", message.lower()))
    if words & COMPLEX or len(words) > 24:
        return "forte", "heurística: tarefa complexa/alto risco"
    if words & SIMPLE and len(words) < 18:
        return "local", "heurística: tarefa curta e delimitada"
    return None, "heurística: caso ambíguo"

def llm_judge(message):
    prompt = (
        "Classifique a tarefa como LOCAL (simples, curta, baixo risco) ou FORTE "
        "(raciocínio, código, estratégia ou risco). Responda só JSON: "
        "{\"route\":\"local|forte\",\"reason\":\"...\"}.\nTarefa: " + message
    )
    result = call_model(JUDGE_MODEL, prompt, "Você é um roteador econômico.")
    clean = result.text.strip().removeprefix("```json").removesuffix("```").strip()
    try:
        payload = json.loads(clean)
        route = "forte" if str(payload["route"]).lower() == "forte" else "local"
        reason = "LLM judge: " + str(payload.get("reason", "classificação sem motivo"))
    except (json.JSONDecodeError, KeyError, TypeError):
        route, reason = "forte", "fallback seguro: JSON inválido do judge"
    return route, reason, result

def choose_route(message):
    route, reason = heuristic(message)
    if route: return route, reason, None
    return llm_judge(message)

## 7. Cenário roteado — mesmas mensagens, mesma ordem

In [8]:
routed_rows, history = [], []
for message in MESSAGES:
    route, reason, judge = choose_route(message)
    model = LOCAL_MODEL if route == "local" else STRONG_MODEL
    result = call_model(model, conversation_prompt(history, message))
    judge_cost = judge.cost_usd if judge else 0.0
    judge_tokens = judge.input_tokens + judge.output_tokens if judge else 0
    routed_rows.append({
        "cenário": "Com roteamento", "mensagem": message, "rota": model,
        "motivo": reason, "input_tokens": result.input_tokens,
        "output_tokens": result.output_tokens, "custo_modelo_usd": result.cost_usd,
        "custo_judge_usd": judge_cost, "tokens_judge": judge_tokens,
    })
    history.append(message)
print(f"Cenário roteado concluído: {len(routed_rows)} chamadas.")

Cenário roteado concluído: 20 chamadas.


## 8. Comparativo consolidado de custo

In [9]:
import pandas as pd

detail = pd.DataFrame(baseline_rows + routed_rows)
detail["chamadas_local"] = (detail["rota"] == LOCAL_MODEL).astype(int)
detail["chamadas_forte"] = (detail["rota"] == STRONG_MODEL).astype(int)
detail["chamadas_judge"] = (detail["tokens_judge"] > 0).astype(int)
detail["tokens_modelo"] = detail["input_tokens"] + detail["output_tokens"]

summary = detail.groupby("cenário", as_index=False).agg(
    mensagens=("mensagem", "count"),
    chamadas_local=("chamadas_local", "sum"),
    chamadas_forte=("chamadas_forte", "sum"),
    chamadas_judge=("chamadas_judge", "sum"),
    tokens_modelo=("tokens_modelo", "sum"),
    tokens_judge=("tokens_judge", "sum"),
    custo_modelos_usd=("custo_modelo_usd", "sum"),
    custo_judge_usd=("custo_judge_usd", "sum"),
)
summary["tokens_totais"] = summary["tokens_modelo"] + summary["tokens_judge"]
summary["custo_total_usd"] = summary["custo_modelos_usd"] + summary["custo_judge_usd"]
summary["custo_médio_por_mensagem_usd"] = summary["custo_total_usd"] / summary["mensagens"]

base_cost = summary.loc[summary["cenário"] == "Sem roteamento", "custo_total_usd"].iloc[0]
summary["economia_vs_sem_roteamento_%"] = (1 - summary["custo_total_usd"] / base_cost) * 100

display(summary.round({
    "custo_modelos_usd": 6, "custo_judge_usd": 6,
    "custo_total_usd": 6, "custo_médio_por_mensagem_usd": 6,
    "economia_vs_sem_roteamento_%": 2,
}))
print("O custo do LLM judge já está incluído no custo total.")

,cenário,mensagens,chamadas_local,chamadas_forte,chamadas_judge,tokens_modelo,tokens_judge,custo_modelos_usd,custo_judge_usd,tokens_totais,custo_total_usd,custo_médio_por_mensagem_usd,economia_vs_sem_roteamento_%
0,Com roteamento,20,10,10,2,9519,204,0.056736,0.000034,9723,0.05677,0.002839,8.42
1,Sem roteamento,20,0,20,0,7605,0,0.061990,0.000000,7605,0.06199,0.003100,0.00


O custo do LLM judge já está incluído no custo total.
